In [88]:
# if you need to install basemap, you can do it using the following command:
# !pip install basemap

In [ ]:
import pandas as pd # data-frame manipulation
from tqdm.auto import tqdm # progress bar
import matplotlib.pyplot as plt # plotting
import seaborn as sns # plotting
import numpy as np # linear algebra
from mpl_toolkits.basemap import Basemap # map plotting
import requests # fetching data from the internet
from scipy import stats # statistical tests

# set the plot style
sns.set(context='paper', style='ticks',
        font_scale=1, palette='colorblind')

## 0. Question

Today, we will answer the following question:

*Does the average temperature affect the colexification of SNOW and ICE in the languages of the world?*

Yesterday, we looked at [this paper](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0151138), where the authors tested whether languages spoken in the colder climates are more likely to colexify the senses ICE and SNOW than the languages spoken in the warmer climates. However, they have only looked at 166 languages, while we have the possibility to test this hypothesis on a much larger set of languages from the [Lexibank](https://lexibank.clld.org/) database. The goal of today session would be to reproduce their results on a much larger sample of languages

## 1. Data

### 1.1. Extracting data from Lexibank

First, we will need to extract data from Lexibank. It has been conveniently gathered for you by the creators of this database, so you can look at it below:

In [ ]:
all_lexibank = pd.read_csv('https://raw.githubusercontent.com/alexeykosh/intro-to-ling-2026/refs/heads/main/s8/lexibank_clipped.csv',
                           low_memory=False, index_col=0)
all_lexibank.head(5)

Let's try to make sense of different columns first:

In [ ]:
all_lexibank.columns

What columns do you think we will need to consider for testing the hypothesis in question? 

How many unique languages are there?

In [ ]:
##################
# YOUR CODE HERE #
##################

How many linguistc families?

In [ ]:
##################
# YOUR CODE HERE #
##################

Can you find data on French? Does the data make sense for you?

In [ ]:
##################
# YOUR CODE HERE #
##################

Can you exclude languages that do not have a family assigned to them?

In [ ]:
##################
# YOUR CODE HERE #
##################

Ok, now let's focus on our hypothesis. We are interested in two senses from lexibank, namely SNOW and ICE. Let's store them in a list as strings:

In [ ]:
terms_for_snow = [
    "ICE", "SNOW"
]

We will not need all the columns, but we will need the following four:

In [ ]:
all_lexibank = all_lexibank[['Form', 'Concepticon_Gloss', 'Glottocode', 'Family']]

Get the subset of the forms from Lexibank that contain words for senses SNOW and ICE. Save it in the variable `subset_si`.

In [ ]:
##################
# YOUR CODE HERE #
##################

Great, now we have the subset of words

In [ ]:
subset_si.head(5)

Let's look at a language that has two separate forms:

In [ ]:
subset_si.query('Glottocode == "russ1263"')

And just one form for both:

In [ ]:
subset_si.query('Glottocode == "hawa1245"')

Now you need to extract the number of unique terms (unique Form) for every language in the database. **Note that we only need to consider languages that have at least two Concepticon_Gloss entries.** Write your code below to fill the list `lang_colex` with the following entries `[glottocode, number_of_unique_forms]`

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's look at the results. How many languages do we have?

In [ ]:
lang_colex[:5]

In [ ]:
len(lang_colex)

Now let's convert it into a dataframe:

In [ ]:
df_colex = pd.DataFrame(lang_colex)
df_colex.columns = ['glottocode', 'Number_of_terms']

Let's look at the distibution of the number of terms in the df_colex dataframe:

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_colex['Number_of_terms'],
         bins=range(1, 7),
         align='left')
sns.despine()
plt.show()

Let's look at the languages that have more than two terms. Is there something interesting that you can notice?

In [ ]:
##################
# YOUR CODE HERE #
##################

Normally, the language with the highest number of terms should be Romanian, so let's look a the terms in Romanian -- does the data look good?

In [ ]:
subset_si.query('Glottocode == "roma1327"')

Finally, we need to add lattitudes and longitudes from glottolog, Let's load the data first:

In [ ]:
glottolog = pd.read_csv('https://raw.githubusercontent.com/alexeykosh/intro-to-ling-2026/refs/heads/main/s8/glottolog.csv')

In [ ]:
glottolog.head(5)

Now merge the two dataframes into a new dataframe by `glottocode` column (use left merge).

In [ ]:
##################
# YOUR CODE HERE #
##################

Check the shape:

In [ ]:
df_merged.shape

Now we need to remove languages that do not have coordinates:

In [ ]:
##################
# YOUR CODE HERE #
##################

Check the results again:

In [ ]:
df_merged.shape

### 1.2. Getting overal temperatures in each language location


We have lattitudes and longitudes for each language in our subset, so we can get average yearly temperatures for their locations. To do this, we need to extract data from NASA on daily temperatures in a certain period, and compute the averages. The code below does that for you. Downloading this data will take a bit of time, so you should not run it. The pre-downloaded data is accesible below.

In [ ]:
def get_overall_avg_temp(lat, lon, start_year, end_year):
    """
    Fetches overall average temperature for a given latitude and longitude using the NASA POWER API.
    Returns a DataFrame with the overall average for the range.
    """
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": f"{start_year}0101",
        "end": f"{end_year}1231",
        "format": "JSON",
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Failed to fetch data for {lat}, {lon}: Status {response.status_code}")
        return None

    # Parse daily temperatures
    data = response.json()['properties']['parameter']['T2M']
    df_temp = pd.DataFrame(list(data.items()), columns=["Date", "Temperature"])
    df_temp["Date"] = pd.to_datetime(df_temp["Date"])
    df_temp.set_index("Date", inplace=True)

    # Calculate overall average temperature for the specified range
    overall_avg = df_temp["Temperature"].mean()
    return pd.DataFrame({
        "latitude": [lat],
        "longitude": [lon],
        "overall_avg_temp": [overall_avg]
    })

# Function to apply to DataFrame
def apply_overall_avg_temp(df, start_year, end_year):
    """
    Applies `get_overall_avg_temp` to each row in a DataFrame with 'latitude' and 'longitude' columns.
    Returns a DataFrame with overall average temperatures for each location.
    """
    results = []
    for _, row in tqdm(df.iterrows()):
        overall_data = get_overall_avg_temp(row['latitude'], row['longitude'], start_year, end_year)
        if overall_data is not None:
            results.append(overall_data)
    return pd.concat(results, ignore_index=True)

# start_year, end_year = 2020, 2025
# df_results = apply_overall_avg_temp(df_merged, start_year, end_year)
# df_results.to_csv('temperature_data.csv')

Let's load the data:

In [ ]:
df_results = pd.read_csv('https://raw.githubusercontent.com/alexeykosh/intro-to-ling-2026/refs/heads/main/s8/temperature_data.csv', index_col=0)

Let's look at the data:

In [ ]:
df_results.head()

What's the max temperature in this dataset? And the minimum?

In [ ]:
##################
# YOUR CODE HERE #
##################

Combine it with our df with lattitudes and longitudes. Use right merge, because we want to keep only the languages for which we have temperature data. Name the new dataframe `df_merged_f`.

In [ ]:
##################
# YOUR CODE HERE #
##################

We have the number of terms for snow and ice, but now we need to convert this into a binary colexification variable. Create a new column, called ``Number_of_terms_bin``, where the value is 1 if there are two or more terms for SNOW and ICE, and 0 if there is only one term.

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's look at the data. We should have data on temperature and colexification for 241 languages. Is there anything interesting that you can notice in this data? Do you see any outliers?

In [ ]:
df_merged_f.glottocode.nunique()

In [ ]:
df_merged_f.head()

## 2. Analysis

Now you need to reproduce the figure from the paper, namely two boxplots for the distribution in average temperatures in the area where the languages with the same or different forms for ICE and SNOW are spoken.

Hint: use [sns.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html)

In [ ]:
##################
# YOUR CODE HERE #
##################

Perform the t-test using ``stats.ttest_ind``. Note that we do not have equal variances, so you need to set the parameter ``equal_var`` to False. 

In [ ]:
##################
# YOUR CODE HERE #
##################

How do you interpret the results? Do they support the hypothesis that languages spoken in colder climates are more likely to colexify SNOW and ICE?

Finally, you need to recreate a map from the paper, showing the coordinates of languages with same or different terms for snow. Use Basemap for this:

In [ ]:
##################
# YOUR CODE HERE #
##################

## 3. Additional analyses

### 3.1. Controlling for geographical location

Let's look at whether the effect will hold if we limit the longitude based on the smallest sample, i.e. the sample of languages where SNOW and ICE are colexified. What is the longitude range for this sample? Store the minimum and maximum longitude in the variables `min_l` and `max_l`.

In [ ]:
##################
# YOUR CODE HERE #
##################

Now limit the longitude range for the other sample to the same range, and reporduce the analysis again. Do you get the same results? What do they suggest about the original paper?

In [ ]:
##################
# YOUR CODE HERE #
##################

In [ ]:
##################
# YOUR CODE HERE #
##################

Now let's do the t-test again, but only for the languages but with this new longitude control. 

In [ ]:
##################
# YOUR CODE HERE #
##################